# Notebook 05 — SHAP plain-English narratives

The Isolation Forest's SHAP panel attributes the model's anomaly score
back to individual engineered features. Raw feature names like
`sensor_9_diff_5` and `sensor_14_roll_std_10` are precise but unreadable
to anyone outside the project. This notebook walks through the
three-stage transform that turns those into something a maintenance
engineer (or interviewer) can follow:

1. **Raw 184-bar SHAP plot** — technically correct, visually illegible.
2. **Top-10 plot with pretty labels** — readable, but still requires
   you to know what "rolling std" means.
3. **One-sentence narrative** — *"Flagged because Sensor 9 is changing
   rapidly and Sensor 14's volatility is rising; the engine is also
   late in its lifecycle."*

The whole point: the model isn't doing anything different at each
stage. Only the **presentation** changes.


## Setup


In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from src.data_loader import load_cmapss, add_rul_to_train, create_anomaly_labels, get_sensor_columns
from src.preprocessing import remove_constant_sensors, train_test_split_by_unit
from src.feature_engineering import build_feature_pipeline
from src.models import IsolationForestDetector
from src.explainability import (
    build_explainer, explain, top_features_for_sample,
    pretty_feature_label, feature_glossary, narrate,
)


## Recreate the data + load the deployed Isolation Forest

Same pipeline as the dashboard. We pick **engine 5**, which spends its
last 30 cycles in the warning zone (`RUL ≤ 30`), so the SHAP panel has
something meaningful to explain.


In [ ]:
train_df, _, _ = load_cmapss('FD001')
train_df = add_rul_to_train(train_df)
train_df = create_anomaly_labels(train_df, threshold=30)
sensor_cols = get_sensor_columns(train_df)
train_df, kept_sensors = remove_constant_sensors(train_df, sensor_cols)

featured = build_feature_pipeline(
    train_df, kept_sensors,
    rolling_windows=[5, 10], lags=[1, 5], ewma_spans=[5]
)
exclude = ['unit_id', 'cycle', 'rul', 'anomaly']
all_feature_cols = [c for c in featured.columns if c not in exclude]

scaler = joblib.load('../models/scaler.pkl')
featured[all_feature_cols] = scaler.transform(featured[all_feature_cols])

iso = IsolationForestDetector()
iso.load('../models/isolation_forest.pkl')

engine_id = 5
engine_df = featured[featured['unit_id'] == engine_id].sort_values('cycle').reset_index(drop=True)
print(f"Engine {engine_id}: {len(engine_df)} cycles, "
      f"last-cycle RUL = {engine_df['rul'].iloc[-1]}")


## Compute SHAP values for engine 5


In [ ]:
healthy_mask = featured['anomaly'] == 0
background = np.nan_to_num(
    featured.loc[healthy_mask, all_feature_cols].values, nan=0.0
)
X_engine = np.nan_to_num(engine_df[all_feature_cols].values, nan=0.0)

explainer = build_explainer(iso, background, max_background=200)
explanation = explain(iso, X_engine, all_feature_cols, explainer=explainer)

print(f"SHAP values shape: {explanation.shap_values.shape}")
print(f"Expected value (baseline score): {explanation.expected_value:.4f}")


## Stage 1 — the raw 184-bar SHAP plot

This is what a naive presentation looks like. Bars for every engineered
feature, sorted by absolute contribution at engine 5's final cycle.
Try reading it.


In [ ]:
final_cycle_idx = len(engine_df) - 1
values = explanation.shap_values[final_cycle_idx]

order = np.argsort(np.abs(values))[::-1]
plt.figure(figsize=(10, 28))
plt.barh(
    np.array(all_feature_cols)[order][::-1],
    values[order][::-1],
    color=['#ef5350' if v > 0 else '#66bb6a' for v in values[order][::-1]],
)
plt.axvline(0, color='gray', linestyle='--', linewidth=0.5)
plt.xlabel('SHAP value (→ anomaly)')
plt.title(f'Stage 1: raw 184-bar SHAP for engine {engine_id}, cycle {int(engine_df["cycle"].iloc[-1])}')
plt.tight_layout()
plt.show()


## Stage 2 — top-10 with plain-English labels

Same SHAP values, but: (1) collapsed to the top 10 by absolute
contribution, and (2) every feature name passed through
`pretty_feature_label` so the y-axis reads like English.

This is what the dashboard's per-cycle drill-down now renders.


In [ ]:
top10 = top_features_for_sample(explanation, final_cycle_idx, k=10)
pretty = [pretty_feature_label(f) for f in top10['feature']]
colors = ['#ef5350' if v > 0 else '#66bb6a' for v in top10['shap_value']]

plt.figure(figsize=(10, 5))
plt.barh(pretty[::-1], top10['shap_value'][::-1], color=colors[::-1])
plt.axvline(0, color='gray', linestyle='--', linewidth=0.5)
plt.xlabel('SHAP value (→ anomaly)')
plt.title(f'Stage 2: top-10 with pretty labels (engine {engine_id}, final cycle)')
plt.tight_layout()
plt.show()


## Stage 3 — the one-sentence narrative

`narrate()` groups the top **positive** (anomaly-pushing) SHAP
contributors by base sensor, describes each one's dominant family in
plain English, and adds a lifecycle clause if `cycle_norm` is also a
strong contributor.


In [ ]:
sentence = narrate(explanation, final_cycle_idx, k=5)
print(sentence)


## And for a healthy cycle (semantic check)

Run the same three stages on cycle 50 of engine 5 — well inside the
healthy zone (RUL ≈ 220). The narrative still produces a "Flagged
because..." sentence here, because **`narrate()` describes what's
pushing the model's score upward at any sample, regardless of whether
that score actually crossed the alarm threshold**. That's the honest
semantics: SHAP explains the *score*, not the binary verdict.

In the dashboard this matters less because users only open the SHAP
panel on cycles they're already investigating. In an interview, it's
worth being precise: the narrative answers *"what is the model paying
attention to here?"* — not *"is this an anomaly?"*. The model's
threshold and the comparison panel answer the latter.


In [ ]:
healthy_cycle_idx = 49  # cycle 50 (0-indexed)
print(f"Stage 3 (healthy cycle 50):")
print(narrate(explanation, healthy_cycle_idx, k=5))


## Feature glossary

For users who want to know what each feature family *means* without
chasing the source code, the dashboard surfaces this as an expander
under the SHAP panel.


In [ ]:
feature_glossary()


## Takeaway

The model's behaviour didn't change between the three stages — every
SHAP value is the same number. What changed is the **interface**
between the model and a human reader. That distinction is what
explainability work mostly is: the science is upstream (build a model
that's honest about why it fires), but the *delivery* is what makes it
land in a meeting or a maintenance shift.
